# CODE TO WEBSCRAPE FROM KICKSTARTER.com
1) first of all download all the necessary libraries (there are some duplicates but I am too lazy to remove them )

In [9]:
import json
import re
import html
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time 
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager




# Kickstarter dataset 


### First try 
I first try to use a dataset from kaggle that includes a lot of projects from kickstarter. The problem as you can clearly see here is that the description of the project is not available, the only thing available is the 'blurb' which I assume is like a very short one-line description of the project. I think it doesn't really make sense to use those for our project since as you can see in the code below, the mean length of those strings is about 18-19 letters, which seems too low for the NLP analysis we want to do

In [2]:
file = pd.read_csv('kickstarter_data_with_features.csv')

C:\Users\simon\AppData\Local\Temp\ipykernel_35896\3137664292.py:1: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  file = pd.read_csv('kickstarter_data_with_features.csv')


Using the Blurb is probably not enough 

In [7]:
print(file['blurb'][0])
file['blurb'].str.split().str.len().mean()

MTS ASL Curriculum Workbook is a reproducible study book to build Early Literacy and Academic skills for primary school age children.


np.float64(18.99379424027926)

### SECOND TRY 
Since the dataset doesn't work, I tried a different thing. I first tried to make a loop that basically went to the kickstarter.com website, looked at the popular page, scraped the descriptions of the projects that are on that first page, then goes to page 2 and does the same, then to page 3 and so on until it reaches n amount of descriptions saved. The problem with that was that I wasn't able to construct a loop that switched to the second page under popular and selected those projects, the loop kept only showing the first page and so only loading up to like 12 projects. 

### FINAL IDEA 

Since the above approach didn't work, I found a website that takes monthly screenshots of the whole website, the page is 'https://webrobots.io/kickstarter-datasets/', there you can find and download the zip file that contains the screenshot of the webiste for the month. In that zip there are a bunch of CSVs that need to be put together (they are different file because putting them all in one single CSV would be too large). So we first get those and put them all together and from that dataset we only want the links (these are the links to the official kickstarter page for each project), then we use these links in a loop that extracts the descriptions by webscraping and also gets the % of how much each project got funded. 


In this case just to test the code I first selected randomly 100 urls from the dataset and then webscraped the information for those

In [ ]:
import pandas as pd
import glob


folder_path = "Kickstarter_2026-03-12T03_20_26_556Z"

files = glob.glob(f"{folder_path}/*.csv")

print("Number of files found:", len(files))

dfs = []

for file in files:
    print("Loading:", file)
    df_temp = pd.read_csv(file)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Final shape:", df.shape)

In [ ]:
df[df['country_displayable_name'] == 'the United States']


In [ ]:


def extract_project_url(x):
    if pd.isna(x):
        return None
    
    try:
       
        d = json.loads(x)
        return d.get("web", {}).get("project")
    except:
        return None
    
df["project_url"] = df[url_col].apply(extract_project_url)

print(df["project_url"].notna().sum())
print(df["project_url"].dropna().head())

In [ ]:
df_valid = df[df["project_url"].notna()]

print("Valid URLs:", len(df_valid))

sample_100 = df_valid.sample(100, random_state=42)
project_urls = sample_100["project_url"].tolist()
project_urls

Here below is the example code run on only one single url

In [ ]:


def scrape_kickstarter_description(url: str) -> dict:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CLASS_NAME, "rte__content"))
        )

        html_source = driver.page_source
        soup = BeautifulSoup(html_source, "html.parser")

        
        description_div = soup.find("div", class_="rte__content")
        description = (
            description_div.get_text(separator=" ", strip=True)
            if description_div else None
        )

        
        title = soup.find("h1", class_="project-name")
        title = title.get_text(strip=True) if title else None

        funding = {
            "pledged": None,
            "usd_pledged": None,
            "converted_pledged_amount": None,
            "goal": None,
            "currency": None
        }

        for script in soup.find_all("script"):
            script_text = script.string or script.get_text()
            if "window.current_project" not in script_text:
                continue

            match = re.search(
                r'window\.current_project\s*=\s*"(.+?)";',
                script_text,
                re.DOTALL
            )
            if not match:
                continue

            raw = match.group(1)

            raw = html.unescape(raw)


            raw = raw.encode("utf-8").decode("unicode_escape")

            try:
                project_data = json.loads(raw)
                funding = {
                    "pledged": project_data.get("pledged"),
                    "usd_pledged": project_data.get("usd_pledged"),
                    "converted_pledged_amount": project_data.get("converted_pledged_amount"),
                    "goal": project_data.get("goal"),
                    "currency": project_data.get("currency")
                }
                break
            except json.JSONDecodeError:
                pass

    finally:
        driver.quit()

    return {
        "url": url,
        "title": title,
        "description": description,
        **funding
    }


url = "https://www.kickstarter.com/projects/addyvalentine/gun-nose"
result = scrape_kickstarter_description(url)

print("Title:", result["title"])
print("Description length:", len(result["description"]) if result["description"] else 0)
print("Description preview:", result["description"][:300] if result["description"] else "None")
print("Pledged:", result["pledged"])
print("USD pledged:", result["usd_pledged"])
print("Converted pledged amount:", result["converted_pledged_amount"])
print("Goal:", result["goal"])
print("Currency:", result["currency"])

Title: GUN NOSE
Description length: 7868
Description preview: Featured on GizmoCrowd – Top Crowdfunding Campaigns A lot of new rewards and tiers have been added to the project since launch. So the above chart should help make the rewards for each tier a little more clear! ⚫ STRETCH GOALS ⚫ THE GAME GUN NOSE is a top-down action-mystery game set in the handcraf
Pledged: 376955.0
USD pledged: 376955.0
Converted pledged amount: 328016
Goal: 35000.0
Currency: USD


here is the code to actually run the loop for all the urls you extracted in project_urls 

In [ ]:

def scrape_kickstarter_description(url: str) -> dict:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    try:
        driver.get(url)

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        time.sleep(2)
        html_source = driver.page_source
        soup = BeautifulSoup(html_source, "html.parser")

        
        description_div = soup.find("div", class_="rte__content")
        description = (
            description_div.get_text(separator=" ", strip=True)
            if description_div else None
        )

        title = soup.find("h1", class_="project-name")
        if not title:
            title = soup.find("h2")
        title = title.get_text(strip=True) if title else None

        funding = {
            "pledged": None,
            "usd_pledged": None,
            "converted_pledged_amount": None,
            "goal": None,
            "currency": None
        }

        for script in soup.find_all("script"):
            script_text = script.string or script.get_text()
            if "window.current_project" not in script_text:
                continue

            match = re.search(
                r'window\.current_project\s*=\s*"(.+?)";',
                script_text,
                re.DOTALL
            )
            if not match:
                continue

            raw = match.group(1)
            raw = html.unescape(raw)

            try:
                raw = raw.encode("utf-8").decode("unicode_escape")
            except Exception:
                pass

            try:
                project_data = json.loads(raw)
                funding = {
                    "pledged": project_data.get("pledged"),
                    "usd_pledged": project_data.get("usd_pledged"),
                    "converted_pledged_amount": project_data.get("converted_pledged_amount"),
                    "goal": project_data.get("goal"),
                    "currency": project_data.get("currency")
                }
                break
            except json.JSONDecodeError:
                pass

        return {
            "url": url,
            "title": title,
            "description": description,
            **funding
        }

    except Exception as e:
        return {
            "url": url,
            "title": None,
            "description": None,
            "pledged": None,
            "usd_pledged": None,
            "converted_pledged_amount": None,
            "goal": None,
            "currency": None,
            "error": str(e)
        }

    finally:
        driver.quit()

In [10]:
rows = []

for i, url in enumerate(project_urls, start=1):
    print(f"Scraping {i}/{len(project_urls)}: {url}")
    row = scrape_kickstarter_description(url)
    rows.append(row)
    time.sleep(1.5)

scraped_df = pd.DataFrame(rows)

print(scraped_df.head())
print(scraped_df.shape)

Scraping 1/100: https://www.kickstarter.com/projects/336157425/wohnraum-schaffen
Scraping 2/100: https://www.kickstarter.com/projects/163030722/lilies
Scraping 3/100: https://www.kickstarter.com/projects/sosomimi/soso-mimi-the-art-of-emotions
Scraping 4/100: https://www.kickstarter.com/projects/jazlink/sirsasana-headstand-burning-man-2022
Scraping 5/100: https://www.kickstarter.com/projects/hudsonvalleyminifarm/mini-farm-barn-revival
Scraping 6/100: https://www.kickstarter.com/projects/kodywachler/kodys-new-single
Scraping 7/100: https://www.kickstarter.com/projects/lisamarieart/lisa-marie-evans-artist-residency-in-budapest
Scraping 8/100: https://www.kickstarter.com/projects/1522564292/high-school-never-ends-bowling-for-soup-jukebox-mu
Scraping 9/100: https://www.kickstarter.com/projects/1153774170/childrens-mobile-boutique-renovation-and-expansion
Scraping 10/100: https://www.kickstarter.com/projects/vidarefanzine/idiot-child-the-first-breath-is-the-beginning-of-death-lp
Scraping 11/

saving them in a dataset:

In [11]:
scraped_df.to_csv("kickstarter_scraped_100.csv", index=False)

In [20]:
scraped_df["description_length"] = scraped_df["description"].str.len()

mean_length = scraped_df["description_length"].mean()

print(mean_length)

745.6477272727273
